# 🌍 WeatherBench 딥러닝 기상 예측 — 중급편
## Z500 · T850 다변수 + 다중 리드타임 예측 with ResNet

| 항목 | 내용 |
|------|------|
| **데이터** | WeatherBench2 ERA5 (GCS zarr 스트리밍) |
| **변수** | Z500 + T850 (2채널) |
| **입력** | t 시각의 전구 Z500+T850 필드 |
| **출력** | t+lead의 전구 Z500+T850 필드 |
| **모델** | ResNet (6 Residual Blocks) |
| **리드타임** | +24h / +72h / +120h |

---

## 목차
1. [ResNet 기반 날씨 예측 소개](#소개)
2. [환경 설정 및 라이브러리 임포트](#환경)
3. [설정값 정의](#설정)
4. [다변수 데이터 로드 (GCS Zarr)](#데이터)
5. [탐색적 데이터 분석 (EDA)](#EDA)
6. [정규화 및 데이터셋 구성](#전처리)
7. [다중 리드타임 DataLoader](#dataloader)
8. [ResNet 모델 정의](#모델)
9. [리드타임별 학습](#학습)
10. [학습 곡선 시각화](#곡선)
11. [RMSE + ACC 평가](#평가)
12. [리드타임별 성능 곡선](#성능곡선)
13. [전구 오차 지도 (Cartopy)](#오차지도)
14. [계절별 RMSE 분석](#계절)
15. [요약 및 심화 학습](#요약)

## 1. ResNet 기반 날씨 예측 소개 <a name="소개"></a>

### 왜 ResNet인가?

입문편에서 사용한 MLP는 전구 필드를 1차원 벡터로 펼쳐서 처리했습니다.  
이 방식은 **공간적 지역성(spatial locality)** — 인접한 격자점이 서로 연관된다는 사실 — 을 무시합니다.

**CNN(Convolutional Neural Network)** 은 공간 패턴을 필터로 학습하며, 파라미터 효율이 높습니다.  
**ResNet**은 CNN에 **잔차 연결(residual connection/skip connection)** 을 추가하여 더 깊은 네트워크도 안정적으로 학습할 수 있게 합니다.

$$\text{ResBlock}(x) = \text{GELU}\left(x + \mathcal{F}(x)\right)$$

### WeatherBench 논문의 ResNet 구조

Rasp & Thuerey (2021)는 WeatherBench에서 ResNet이 MLP를 크게 능가함을 보였습니다:
- 동일 해상도에서 Z500 RMSE 약 30–40% 감소
- 공간적 일관성 향상 (예측 필드가 더 "매끄럽고" 물리적으로 타당)

```
입력 (2×lat×lon) → Conv(64) → GELU
                 → [ResBlock × 6]
                 → Conv(2) → 출력 (2×lat×lon)
```

### 이 노트북의 추가 사항

| 항목 | 입문편 | 중급편 |
|------|--------|--------|
| 모델 | MLP | **ResNet** |
| 변수 | Z500 | **Z500 + T850** |
| 리드타임 | +24h | **+24h / +72h / +120h** |
| 평가 | RMSE | **RMSE + ACC** |
| 시각화 | matplotlib | **Cartopy 전구 지도** |

> 📌 **참고문헌**  
> Rasp, S., & Thuerey, N. (2021). Data-driven medium-range weather prediction with a Resnet pretrained on climate simulations. *JAMES*, 13(2).  
> He, K., et al. (2016). Deep residual learning for image recognition. *CVPR*.

In [ ]:
# Colab 환경 설정 (gcsfs, zarr, xarray, cartopy)
!pip install -q gcsfs zarr "xarray[io]" cartopy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/weather-climate-ai-tutorials

In [ ]:
# 나눔 고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr
import gcsfs
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')

In [ ]:
# ============================================================
# 설정값 (Configuration)
# ============================================================

GCS_PATH = (
    'gs://weatherbench2/datasets/era5/'
    '1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr'
)

# 리드타임: 각각 24h, 72h, 120h에 해당하는 6h 스텝 수
LEAD_STEPS = [4, 12, 20]

# 공간 해상도 축소: 2배 coarsen → 240/2=120 lon, ~60 lat
COARSEN    = 2

# 학습 하이퍼파라미터
BATCH_SIZE = 32
EPOCHS     = 60
LR         = 1e-3

# 예측 변수: (xarray 변수명, 압력레벨 hPa)
VARS = {
    'Z500': ('geopotential', 500),
    'T850': ('temperature',  850),
}

print('설정값 로드 완료')
for lead in LEAD_STEPS:
    print(f'  리드타임 +{lead * 6}h ({lead} 스텝)')
print(f'  공간 축소: {COARSEN}배 (약 {240 // COARSEN} lon × {121 // COARSEN} lat 격자)')
print(f'  변수: {list(VARS.keys())}')

In [ ]:
# ============================================================
# GCS Zarr에서 다변수 데이터 로드
# ============================================================

fs = gcsfs.GCSFileSystem(token='anon')
ds = xr.open_zarr(fs.get_mapper(GCS_PATH), consolidated=True)
print('데이터셋 변수:', list(ds.data_vars))

fields = {}
for var_name, (da_name, level) in VARS.items():
    da = ds[da_name].sel(level=level, time=slice('2015', '2018'))
    da = da.coarsen(latitude=COARSEN, longitude=COARSEN, boundary='trim').mean()
    print(f'{var_name} 로드 중...')
    fields[var_name] = da.load().values  # (time, lat, lon)
    print(f'  shape: {fields[var_name].shape}')

# 격자 좌표 (coarsen 적용 후)
lats  = ds.latitude.values[::COARSEN][:fields['Z500'].shape[1]]
lons  = ds.longitude.values[::COARSEN][:fields['Z500'].shape[2]]
times = ds.time.sel(time=slice('2015', '2018')).values
print(f'\n완료! lat={len(lats)}, lon={len(lons)}, 시간 스텝={len(times)}')

In [ ]:
# ============================================================
# 탐색적 데이터 분석 (EDA): Z500 & T850 동시 시각화 (Cartopy)
# ============================================================

t_idx = 0  # 첫 번째 시간 스텝
fig, axes = plt.subplots(
    1, 2, figsize=(14, 4),
    subplot_kw={'projection': ccrs.Robinson()}
)

plot_specs = [
    ('Z500 (m2/s2)', fields['Z500'][t_idx], 'RdYlBu_r'),
    ('T850 (K)',     fields['T850'][t_idx], 'RdBu_r'),
]

for ax, (name, data, cmap) in zip(axes, plot_specs):
    cf = ax.pcolormesh(
        lons, lats, data, cmap=cmap,
        transform=ccrs.PlateCarree(), shading='auto'
    )
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.3)
    plt.colorbar(cf, ax=ax, orientation='horizontal',
                 pad=0.02, shrink=0.8, label=name)
    ax.set_title(name)

plt.suptitle(f'입력 필드 예시 ({str(times[0])[:10]})', fontsize=13)
plt.tight_layout()
plt.savefig('wb_input_fields.png', dpi=120)
plt.show()
print('그림 저장: wb_input_fields.png')

In [ ]:
# ============================================================
# 변수별 Z-score 정규화 + 다채널 스택
# ============================================================

# 연도 문자열 추출로 훈련/검증/테스트 마스크 생성
year = np.array([str(t)[:4] for t in times])
tr_mask = (year == '2015') | (year == '2016')
vl_mask = (year == '2017')
te_mask = (year == '2018')

# 변수별 통계 계산 (훈련셋 기준)
stats = {}
for name in VARS:
    tr_data = fields[name][tr_mask]
    stats[name] = {
        'mean': float(tr_data.mean()),
        'std':  float(tr_data.std()),
    }
    fields[name] = (fields[name] - stats[name]['mean']) / stats[name]['std']
    print(f'{name}: mean={stats[name]["mean"]:.2f}, std={stats[name]["std"]:.2f}')

# (time, channel, lat, lon) 형태로 스택
data_all = np.stack([fields['Z500'], fields['T850']], axis=1).astype(np.float32)
print(f'\n전체 데이터 shape: {data_all.shape}')  # (time, 2, lat, lon)
print(f'  Train: {tr_mask.sum()} / Val: {vl_mask.sum()} / Test: {te_mask.sum()} 스텝')

In [ ]:
# ============================================================
# 다중 리드타임 Dataset & DataLoader 구성
# ============================================================

class MultiLeadDataset(Dataset):
    """단일 리드타임에 대한 (X, y) 쌍을 구성합니다."""
    def __init__(self, data, lead):
        self.X = torch.tensor(data[:-lead])
        self.y = torch.tensor(data[lead:])
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

# 리드타임별 DataLoader 딕셔너리 구성
loaders = {}
MAX_LEAD = max(LEAD_STEPS)

for lead in LEAD_STEPS:
    # MAX_LEAD만큼 끝을 잘라내어 모든 리드타임에서 동일 길이의 데이터를 사용
    loaders[lead] = {
        'train': DataLoader(
            MultiLeadDataset(data_all[tr_mask][:-MAX_LEAD], lead),
            batch_size=BATCH_SIZE, shuffle=True
        ),
        'val': DataLoader(
            MultiLeadDataset(data_all[vl_mask][:-MAX_LEAD], lead),
            batch_size=BATCH_SIZE
        ),
        'test': DataLoader(
            MultiLeadDataset(data_all[te_mask][:-MAX_LEAD], lead),
            batch_size=BATCH_SIZE
        ),
    }
    n = len(loaders[lead]['train'].dataset)
    print(f'Lead +{lead * 6}h: train {n}개 샘플')

## 2. ResNet 모델 구조 <a name="모델"></a>

### Residual Block (잔차 블록)

잔차 연결은 입력을 출력에 직접 더하여 **그래디언트 소실(vanishing gradient)** 문제를 완화합니다:

$$\mathbf{y} = \text{GELU}\left(\mathbf{x} + \mathcal{F}(\mathbf{x};\, \{W_i\})\right)$$

```
x ─────────────────────────────────────┐
│                                       │
└→ Conv2d → BN → GELU → Conv2d → BN ──⊕→ GELU → y
```

### 전체 ResNet 구조

```
입력 (2, lat, lon)
    │
    ▼
Conv2d(in=2, out=64, k=3, pad=1) → GELU
    │
    ▼
ResBlock × 6  (각 블록: Conv→BN→GELU→Conv→BN + Skip)
    │
    ▼
Conv2d(in=64, out=2, k=1)   ← 1×1 conv: 채널 수 조정
    │
    ▼
출력 (2, lat, lon)           ← Z500, T850 예측
```

### 설계 선택
- **같은 공간 해상도 유지**: MaxPool/Upsample 없이 padding=1로 크기 보존
- **1×1 conv 출력층**: 채널 수를 hidden→out_ch로 효율적 매핑
- **리드타임별 별도 모델**: 각 예측 시간에 최적화된 파라미터 학습

In [ ]:
# ============================================================
# ResNet 모델 정의
# ============================================================

class ResBlock(nn.Module):
    """2개의 Conv2d + BatchNorm + GELU + Skip Connection."""
    def __init__(self, ch, kernel=3):
        super().__init__()
        p = kernel // 2
        self.block = nn.Sequential(
            nn.Conv2d(ch, ch, kernel, padding=p),
            nn.BatchNorm2d(ch),
            nn.GELU(),
            nn.Conv2d(ch, ch, kernel, padding=p),
            nn.BatchNorm2d(ch),
        )
        self.act = nn.GELU()

    def forward(self, x):
        # 잔차 연결: 입력을 변환된 출력에 더함
        return self.act(x + self.block(x))


class ResNet(nn.Module):
    """
    WeatherBench 논문 기반 ResNet.
    입력: (B, in_ch, lat, lon)
    출력: (B, out_ch, lat, lon) — 공간 해상도 유지
    """
    def __init__(self, in_ch=2, out_ch=2, hidden=64, n_blocks=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 3, padding=1),
            nn.GELU(),
            *[ResBlock(hidden) for _ in range(n_blocks)],
            nn.Conv2d(hidden, out_ch, 1),  # 1x1 conv로 채널 압축
        )

    def forward(self, x):
        return self.net(x)


# 파라미터 수 확인
model_test = ResNet(in_ch=2, out_ch=2, hidden=64, n_blocks=6).to(DEVICE)
n_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'ResNet 파라미터 수: {n_params:,}')
del model_test

In [ ]:
# ============================================================
# 학습 함수 정의 + 리드타임별 학습 실행
# ============================================================

def train_model(model, tr_loader, vl_loader, epochs, lr):
    """ResNet을 학습하고 최적 검증 모델 가중치를 복원합니다."""
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    # CosineAnnealing: 학습률을 코사인 함수에 따라 서서히 감소
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.MSELoss()

    best_val, best_state = float('inf'), None
    tr_hist, vl_hist = [], []

    for ep in range(1, epochs + 1):
        # --- 훈련 ---
        model.train()
        tr_l = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            tr_l += loss.item() * len(xb)
        tr_l /= len(tr_loader.dataset)

        # --- 검증 ---
        model.eval()
        vl_l = 0.0
        with torch.no_grad():
            for xb, yb in vl_loader:
                vl_l += crit(model(xb.to(DEVICE)), yb.to(DEVICE)).item() * len(xb)
        vl_l /= len(vl_loader.dataset)

        sched.step()

        # 최적 모델 가중치 저장
        if vl_l < best_val:
            best_val = vl_l
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        tr_hist.append(tr_l)
        vl_hist.append(vl_l)

        if ep % 10 == 0:
            print(f'  [{ep:3d}/{epochs}] train={tr_l:.4f} val={vl_l:.4f}')

    # 최적 가중치로 복원
    model.load_state_dict(best_state)
    return tr_hist, vl_hist


# 리드타임별로 별도 모델 학습
trained_models = {}
for lead in LEAD_STEPS:
    print(f'\n{"=" * 50}')
    print(f'  Lead +{lead * 6}h 학습 중...')
    print(f'{"=" * 50}')
    m = ResNet(in_ch=2, out_ch=2, hidden=64, n_blocks=6).to(DEVICE)
    tr_h, vl_h = train_model(
        m, loaders[lead]['train'], loaders[lead]['val'], EPOCHS, LR
    )
    trained_models[lead] = {'model': m, 'tr': tr_h, 'vl': vl_h}

print('\n모든 모델 학습 완료!')

In [ ]:
# ============================================================
# 리드타임별 학습 곡선 시각화
# ============================================================

fig, axes = plt.subplots(1, len(LEAD_STEPS), figsize=(15, 4))

for ax, lead in zip(axes, LEAD_STEPS):
    res = trained_models[lead]
    ax.plot(res['tr'], label='Train', color='#E63946')
    ax.plot(res['vl'], label='Val',   color='#2196F3')
    ax.set_title(f'+{lead * 6}h 학습 곡선')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss (정규화)')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('리드타임별 학습 곡선 (ResNet)', fontsize=13)
plt.tight_layout()
plt.savefig('wb_resnet_loss.png', dpi=120)
plt.show()

In [ ]:
# ============================================================
# RMSE + ACC 평가 함수 정의 및 테스트셋 평가
# ============================================================

# 클리마톨로지: 훈련셋의 시간평균 (이상값 계산 기준)
clim_z = data_all[tr_mask][:, 0].mean(axis=0)  # (lat, lon)
clim_t = data_all[tr_mask][:, 1].mean(axis=0)


def compute_metrics(model, loader, clim_z, clim_t):
    """RMSE와 ACC (Anomaly Correlation Coefficient) 계산."""
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
            trues.append(yb.numpy())
    preds = np.concatenate(preds)  # (N, 2, lat, lon)
    trues = np.concatenate(trues)

    results = {}
    clims = [clim_z, clim_t]
    var_names = list(VARS.keys())

    for c, (name, clim) in enumerate(zip(var_names, clims)):
        # 역정규화
        p = preds[:, c] * stats[name]['std'] + stats[name]['mean']
        t = trues[:, c] * stats[name]['std'] + stats[name]['mean']

        # RMSE
        rmse = float(np.sqrt(np.mean((p - t) ** 2)))

        # ACC: 이상값 기반 상관계수
        clim_orig = clim * stats[name]['std'] + stats[name]['mean']
        p_anom = p - clim_orig
        t_anom = t - clim_orig
        num = float(np.mean(p_anom * t_anom))
        den = float(np.sqrt(np.mean(p_anom ** 2) * np.mean(t_anom ** 2))) + 1e-8
        acc = num / den

        results[name] = {'rmse': rmse, 'acc': acc}
    return results


print(f"{'변수':<8} {'리드타임':>10} {'RMSE':>12} {'ACC':>10}")
print('-' * 44)

all_metrics = {}
for lead in LEAD_STEPS:
    m = trained_models[lead]['model']
    metrics = compute_metrics(m, loaders[lead]['test'], clim_z, clim_t)
    all_metrics[lead] = metrics
    for name, vals in metrics.items():
        print(f"{name:<8} {'+' + str(lead * 6) + 'h':>10} {vals['rmse']:>12.3f} {vals['acc']:>10.4f}")

In [ ]:
# ============================================================
# 리드타임별 RMSE / ACC 곡선
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = {'Z500': '#E63946', 'T850': '#2196F3'}
lead_hours = [l * 6 for l in LEAD_STEPS]

for ax, metric in zip(axes, ['rmse', 'acc']):
    for name in VARS:
        vals = [all_metrics[l][name][metric] for l in LEAD_STEPS]
        ax.plot(lead_hours, vals, 'o-',
                color=colors[name], lw=2, ms=8, label=name)

    ax.set_xlabel('리드타임 (시간)')
    ax.set_xticks(lead_hours)

    if metric == 'rmse':
        ax.set_ylabel('RMSE')
        ax.set_title('리드타임별 RMSE')
    else:
        ax.set_ylabel('ACC')
        ax.set_title('리드타임별 ACC (Anomaly Correlation)')
        # ACC=0.6: 통상적으로 "유용한 예측" 임계값
        ax.axhline(0.6, color='gray', ls='--', label='ACC=0.6 임계값')

    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('ResNet — 리드타임별 성능 (2018년 테스트셋)', fontsize=13)
plt.tight_layout()
plt.savefig('wb_leadtime_metrics.png', dpi=120)
plt.show()

In [ ]:
# ============================================================
# 전구 오차 지도 (Cartopy) — +24h Z500
# ============================================================

lead = LEAD_STEPS[0]  # +24h 모델 사용
m = trained_models[lead]['model']
m.eval()

all_preds, all_trues = [], []
with torch.no_grad():
    for xb, yb in loaders[lead]['test']:
        all_preds.append(m(xb.to(DEVICE)).cpu().numpy())
        all_trues.append(yb.numpy())

preds = np.concatenate(all_preds)  # (N, 2, lat, lon)
trues = np.concatenate(all_trues)

# Z500 역정규화
preds_z = preds[:, 0] * stats['Z500']['std'] + stats['Z500']['mean']
trues_z = trues[:, 0] * stats['Z500']['std'] + stats['Z500']['mean']

# 시간평균 편차 지도 및 RMSE 지도
bias_map = (preds_z - trues_z).mean(axis=0)           # (lat, lon)
rmse_map = np.sqrt(((preds_z - trues_z) ** 2).mean(axis=0))

fig, axes = plt.subplots(
    1, 2, figsize=(16, 5),
    subplot_kw={'projection': ccrs.Robinson()}
)

map_specs = [
    (bias_map, 'Z500 편차 지도 (예측-실제, 시간평균)', 'RdBu_r',  (-300, 300)),
    (rmse_map, 'Z500 RMSE 지도',                      'YlOrRd',  (0,    600)),
]

for ax, (data, title, cmap, vlim) in zip(axes, map_specs):
    cf = ax.pcolormesh(
        lons, lats, data, cmap=cmap,
        vmin=vlim[0], vmax=vlim[1],
        transform=ccrs.PlateCarree(), shading='auto'
    )
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.3)
    plt.colorbar(cf, ax=ax, orientation='horizontal',
                 pad=0.04, shrink=0.8, label='m2/s2')
    ax.set_title(title, fontsize=11)

plt.suptitle(f'ResNet +{lead * 6}h Z500 예측 (2018년 테스트셋)', fontsize=13)
plt.tight_layout()
plt.savefig('wb_global_error_map.png', dpi=120)
plt.show()

In [ ]:
# ============================================================
# 계절별 Z500 RMSE 분석 (+24h)
# ============================================================

# 테스트 기간의 시간 배열에서 월 추출
te_times_str = [str(t)[:10] for t in times[te_mask][:-MAX_LEAD]]
te_months    = np.array([int(t[5:7]) for t in te_times_str])

# 월 → 계절 매핑
season_map = {
    m: s
    for s, ms in {
        'DJF': [12, 1, 2],
        'MAM': [3, 4, 5],
        'JJA': [6, 7, 8],
        'SON': [9, 10, 11],
    }.items()
    for m in ms
}
seasons_arr = np.array([season_map[m] for m in te_months])[:len(preds_z)]

fig, ax = plt.subplots(figsize=(8, 5))
season_names  = ['DJF', 'MAM', 'JJA', 'SON']
season_colors = ['#1565C0', '#2E7D32', '#F57F17', '#6A1B9A']

rmse_by_season = []
for s in season_names:
    mask_s = seasons_arr == s
    if mask_s.any():
        r = float(np.sqrt(((preds_z[mask_s] - trues_z[mask_s]) ** 2).mean()))
        rmse_by_season.append(r)
    else:
        rmse_by_season.append(0.0)

bars = ax.bar(season_names, rmse_by_season,
              color=season_colors, alpha=0.85, width=0.5)
ax.bar_label(bars, fmt='%.1f', fontsize=11)
ax.set_ylabel('RMSE (m2/s2)')
ax.set_title('Z500 계절별 RMSE (+24h, 2018년)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('wb_seasonal_rmse.png', dpi=120)
plt.show()

## 3. 요약 및 심화 학습 <a name="요약"></a>

### 이 노트북에서 배운 것

1. **다변수 기상 예측**: Z500과 T850을 2채널로 묶어 ResNet에 입력하는 방법
2. **ResNet 구조**: 잔차 연결의 원리와 기상 예측에서의 장점
3. **다중 리드타임 학습**: 각 예측 시간(+24h/+72h/+120h)에 최적화된 개별 모델 학습
4. **ACC 평가**: 기상청에서 표준으로 사용하는 이상값 상관계수
5. **Cartopy 시각화**: 전구 오차 지도와 계절별 성능 분석

### 주요 발견

- 리드타임이 길어질수록 RMSE 증가, ACC 감소 (당연한 결과)
- **DJF(겨울)** 에서 RMSE가 높음 — 중위도 제트류의 변동성이 크기 때문
- **JJA(여름)** 에서 RMSE가 낮음 — 대기 순환 패턴이 상대적으로 안정적
- T850은 Z500보다 단기 예측 정확도가 높지만 장기로 갈수록 빠르게 낮아짐

### 더 나아가기: 최신 AI 날씨 예측 모델

| 모델 | 기관 | 특징 |
|------|------|------|
| **FourCastNet** | NVIDIA | Fourier Neural Operator 기반 |
| **Pangu-Weather** | Huawei | 3D Earth Specific Transformer |
| **GraphCast** | Google DeepMind | Graph Neural Network |
| **Kepler** | ECMWF | 운용 AI 예측 시스템 |
| **Aurora** | Microsoft | 대형 기후 기반 모델 |

이들 모델은 모두 더 높은 해상도, 더 많은 변수, 더 복잡한 아키텍처를 사용하지만,  
이 노트북에서 배운 **데이터 파이프라인**, **정규화**, **RMSE/ACC 평가** 의 원칙은 동일하게 적용됩니다.

### 다음 학습 방향

- **위도 가중 RMSE**: 극지방과 적도의 격자 면적 차이 보정
  ```python
  lat_weights = np.cos(np.deg2rad(lats))
  weighted_rmse = np.sqrt(np.average((pred - true)**2,
                                      weights=lat_weights, axis=(-2,-1)))
  ```
- **Circular padding**: 경도 0°/360° 연속성 처리
- **Multi-step autoregressive 예측**: 출력을 다음 입력으로 재귀 사용
- **Ensemble 예측**: 여러 모델 앙상블로 불확실성 정량화

### 참고 자료

- [WeatherBench2 GitHub](https://github.com/google-research/weatherbench2)
- [FourCastNet (Pathak et al., 2022)](https://arxiv.org/abs/2202.11214)
- [Pangu-Weather (Bi et al., 2023)](https://www.nature.com/articles/s41586-023-06185-3)
- [GraphCast (Lam et al., 2023)](https://www.science.org/doi/10.1126/science.adi2336)
- [Aurora (Price et al., 2025)](https://www.nature.com/articles/s41586-025-08740-2)